# 0 Forecasting baselines: which model has to be beaten, and by how much

`01_probabilistic` puts a distribution on a forecast of one Australian state's electricity
demand. Before a forecast gets a width it has to be a forecast, and this notebook builds
that point forecast on the same series and the same window, in the order that keeps it
honest:

1. the window, as an index matrix;
2. split in time first, then window;
3. normalise on the training statistics, and only those;
4. name the naive model your metric is measured against;
5. a linear model and a GRU, trained with `vectormesh.training.Trainer`;
6. a table with a cost column, and the test split, once.

The series is `australian_electricity_demand` from the Monash archive, state `T1`:
half-hourly demand in MW, 2002 to 2015, with a clean daily and weekly pattern.
`australian_electricity.py` loads it and cuts it in calendar time: the last full year is
test, the year before it validation, the rest train.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components.metrics import MAE
from vectormesh.training import Trainer, TrainerSettings

from australian_electricity import load_ausalia
from windowing import window_index
from notebooktester import param

DAY = 48  # half-hourly steps per day
W, H = (
    4 * DAY,
    DAY,
)  # four days of history, one day of horizon: the window of 01_probabilistic
STRIDE = 24  # 12 hours between training windows; validation and test use 2 * STRIDE
EPOCHS = param(8, test=1)
SEEDS = param((0, 1, 2), test=(0,))

## 0.1 The window is an index matrix

A model takes a fixed-length input, and a series is one long vector. Windowing cuts the
vector into many overlapping pieces of `W + H` steps: the first `W` are the history the
model sees, the last `H` the future it has to predict.

The cut is a single indexing operation. Take nine numbers and windows of three:

In [ ]:
x = torch.arange(10, 100, 10)
size = 3
time_axis = torch.arange(size).reshape(1, -1)  # (1, 3): the offsets inside one window
starts = torch.arange(len(x) - size + 1).reshape(
    -1, 1
)  # (7, 1): where each window starts
idx = time_axis + starts  # broadcast (1, 3) + (7, 1) -> (7, 3)
print(idx)
print(x[idx])

Broadcasting adds a `(1, 3)` row of offsets to a `(7, 1)` column of start positions and
returns a `(7, 3)` matrix: row `i` is the index of window `i`. Indexing the series with that
matrix returns all seven windows at once. A series of length `n` holds `n - size + 1`
windows of `size` steps.

`window_index` in `windowing.py` is this broadcast, with a `stride` between start positions:

In [ ]:
print(torch.equal(window_index(len(x), size), idx))
print(window_index(len(x), size, stride=2))

Neighbouring windows at stride 1 share all steps but one. At `W + H = 240` half-hours that
is 239 of 240, so striding by 24 (12 hours) throws away almost no information and makes an
epoch 24 times shorter. Validation and test windows use stride 48, one forecast per day.

## 0.2 Split first, then window

In [ ]:
ausalia = load_ausalia()
print(ausalia)
for name, s in ausalia.splits.items():
    a, b = ausalia.timestamps[s.start], ausalia.timestamps[s.stop - 1]
    print(f"{name:5s} {a:%Y-%m-%d} .. {b:%Y-%m-%d}   {s.stop - s.start:6d} half-hours")

The order matters. Window the whole series first and split the windows afterwards, and
the windows that straddle a boundary hold training steps in their history and validation
steps in their target. That is a few hundred rows out of thousands, too few to show up in
a score, which is what makes it dangerous.

Split first and those windows never exist. Each split loses the `W + H - 1` windows that
would have crossed its edge:

In [ ]:
for name, s in ausalia.splits.items():
    n = s.stop - s.start
    print(
        f"{name:5s} {n - W - H + 1:6d} stride-1 windows inside the split, "
        f"{W + H - 1} lost at its edge"
    )

## 0.3 Normalise on the training statistics, and only those

In [ ]:
train_values = ausalia.values[ausalia.splits["train"]]
mu, sigma = (
    train_values.mean(),
    train_values.std(),
)  # fitted on train, applied everywhere

for name, s in ausalia.splits.items():
    v = ausalia.values[s]
    z = (v - mu) / sigma
    print(
        f"{name:5s} mean {v.mean():7.1f} MW  std {v.std():6.1f}   "
        f"scaled: mean {z.mean():+.3f}  std {z.std():.3f}"
    )

Scaled with the training statistics, the validation year lands at mean -0.351 and the test
year at -0.401, both with a smaller spread than training. Demand in 2013-2015 is lower than
the 2002-2013 average, and that offset is real information about the future. A scaler
fitted on all three splits would have hidden it from the numbers, and every model scored
afterwards would have been scored on data that had already told the scaler where it lies.
Fit the scaler once, on train, and apply it everywhere.

The windows, cut inside each split from the scaled values:

In [ ]:
def windows(split: str, stride: int):
    # scaled with the training statistics, windowed inside one split
    values = (ausalia.values[ausalia.splits[split]] - mu) / sigma
    win = values[window_index(len(values), W + H, stride)]
    return win[:, :W], win[:, W:]


xtr, ytr = windows("train", STRIDE)
xva, yva = windows("val", 2 * STRIDE)
xte, yte = windows("test", 2 * STRIDE)
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}   test {tuple(xte.shape)}")

## 0.4 Which naive model is the baseline?

Two forecasts cost nothing:

- **naive**: repeat the last observed half-hour for all 48 steps;
- **seasonal naive**: repeat the last observed day, `x[:, -DAY:]`, so tomorrow at 18:00
  is today at 18:00.

Demand rises in the morning and falls at night, so a flat line through the last value
misses the shape of every day. The prediction, before measuring it: seasonal naive is the
stronger of the two.

A metric needs the baseline it is measured against. **MASE** divides the MAE by the
in-sample MAE of a naive forecast on the training data, so the number carries its
reference with it. With a seasonal baseline the reference is the seasonal one (Hyndman
and Koehler's seasonal MASE): the mean absolute difference between a training step and
the step one day earlier. Below 1 means better than repeating yesterday was in-sample.

In [ ]:
def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


z_train = (train_values - mu) / sigma
for lag, label in ((1, "one step"), (DAY, "one day")):
    print(
        f"in-sample naive error, lag {label:8s}: {float((z_train[lag:] - z_train[:-lag]).abs().mean()):.4f}"
    )

mase_scale = float(
    (z_train[DAY:] - z_train[:-DAY]).abs().mean()
)  # seasonal MASE, m = DAY

results: dict[str, dict] = {}
results["naive (repeat last)"] = {
    "mae": mae(yva, xva[:, -1:].expand(-1, H)),
    "std": 0.0,
    "params": 0,
    "secs": 0.0,
}
results["seasonal naive (1 day)"] = {
    "mae": mae(yva, xva[:, -DAY:]),
    "std": 0.0,
    "params": 0,
    "secs": 0.0,
}

for name, r in results.items():
    print(f"{name:24s} val MAE {r['mae']:.4f}   MASE {r['mae'] / mase_scale:.3f}")

The prediction holds, by a wide margin. Seasonal naive has a validation MAE of 0.2906
(MASE 0.896); naive has 0.7362 (MASE 2.270), two and a half times worse. In-sample, one
step ahead, repeating the last value is the better guess (0.1294 against 0.3243), but a
day ahead the daily cycle dominates. So the baseline of this notebook is **seasonal naive**,
and "MASE 0.9" means "a little better than repeating yesterday did on the training years".

Without that sentence a MASE is decoration. Scaled by the one-step error instead, every
row of the table below would read about 2.5 times higher, and a model that beats both
baselines would report a MASE above 1.

## 0.5 A linear model

`nn.Linear(W, H)`: every output half-hour is a weighted sum of the 192 input half-hours.
No nonlinearity, no state. The seasonal naive forecast is one particular setting of these
weights (a 1 on the step one day back, 0 elsewhere), so the linear model can only lose to
it by failing to learn.

`vectormesh.training.Trainer` runs the loop: Adam, MSE loss, the validation MAE after every
epoch. One `Trainer` per seed; `progress=False` drops the progress bars.

In [ ]:
validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))


def fit(make_model, seeds=SEEDS, epochs=EPOCHS, batch_size=64, lr=1e-3) -> dict:
    scores, secs, models = [], [], []
    for seed in seeds:
        torch.manual_seed(seed)
        model = make_model()
        trainloader = DataLoader(
            TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
        )
        settings = TrainerSettings(
            epochs=epochs,
            metrics=[MAE()],
            logdir=Path("modellogs").resolve(),
            train_steps=len(trainloader),
            valid_steps=len(validloader),
            optimizer_kwargs={"lr": lr},
            earlystop_kwargs=None,
        )
        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=nn.MSELoss(),
            optimizer=torch.optim.Adam,
            traindataloader=trainloader,
            validdataloader=validloader,
            progress=False,
        )
        t0 = time.time()
        result = trainer.loop()
        secs.append(time.time() - t0)
        scores.append(result.metric_dict["MAE"])
        models.append(model.eval())
    return {
        "mae": float(np.mean(scores)),
        "std": float(np.std(scores)),
        "params": sum(p.numel() for p in models[0].parameters()),
        "secs": float(np.mean(secs)),
        "models": models,
    }


def report(name: str, r: dict) -> None:
    results[name] = r
    print(
        f"{name:24s} val MAE {r['mae']:.4f} +/- {r['std']:.4f}   MASE {r['mae'] / mase_scale:.3f}   "
        f"{r['params']:>7,} params   {r['secs']:5.1f}s"
    )


class Linear(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(W, H)

    def forward(self, x):  # (B, W) -> (B, H)
        return self.fc(x)


report("linear", fit(Linear))

## 0.6 A GRU

A recurrent network walks the window one step at a time and carries a hidden state. A
GRU adds two gates to that state: an update gate that decides how much of the old state
to keep, and a reset gate that decides how much of it the next candidate state may use.

`nn.GRU` returns two things: `out`, the hidden state at every step, `(B, T, hidden)`, and
the final state, `(layers, B, hidden)`. A fixed horizon from a fixed window needs one
summary per window, so the head reads `out[:, -1, :]`, the state after the last step.
This is the trunk `01_probabilistic` puts its Gaussian and quantile heads on.

In [ ]:
class GRU(nn.Module):
    def __init__(self, hidden: int = 64):
        super().__init__()
        self.rnn = nn.GRU(input_size=1, hidden_size=hidden, batch_first=True)
        self.head = nn.Linear(hidden, H)

    def forward(self, x):  # (B, W) -> (B, H)
        out, _ = self.rnn(x.unsqueeze(-1))  # (B, W, 1) -> (B, W, hidden)
        return self.head(out[:, -1, :])


report("GRU(64)", fit(GRU))

## 0.7 The table, with the cost column

In [ ]:
order = sorted(results, key=lambda k: results[k]["mae"])
base = results["seasonal naive (1 day)"]["mae"]
print(
    f"{'model':24s} {'val MAE':>8} {'+/-':>7} {'MASE':>6} {'vs s-naive':>11} {'params':>8} {'secs':>7}"
)
print("-" * 77)
for name in order:
    r = results[name]
    print(
        f"{name:24s} {r['mae']:8.4f} {r['std']:7.4f} {r['mae'] / mase_scale:6.3f} "
        f"{r['mae'] / base:11.3f} {r['params']:8,} {r['secs']:7.1f}"
    )

Read the `+/-` column before the ranking: it is the spread over three seeds, and it says
whether two rows are ordered at all.

- **Both trained models beat seasonal naive.** The linear model cuts its validation MAE by
  18% (0.2381 against 0.2906), the GRU by 13% (0.2529).
- **The linear model beats the GRU.** The gap, 0.0148, is about twice the seed spread of
  either model (0.0059 and 0.0071), so it is a measured ordering on this run, not a coin
  flip, though a narrow one.
- **The cost column runs the other way.** The linear model has 9,264 parameters and trains
  in a fraction of a second; the GRU has 15,984 and takes some twenty to thirty seconds,
  about a hundred times as long, for the worse score. The seconds depend on the machine;
  the ratio is what to read. A tie at a hundred times the cost would be a loss; this is not even
  a tie.

The linear layer has a direct path to "this time yesterday": one weight per (input step,
output step) pair, and seasonal naive is one of its possible settings. The GRU has to carry
that information through 192 recurrent steps and a 64-number state.

In [ ]:
picks = [40, 140, 240]
fig, axes = plt.subplots(1, len(picks), figsize=(14, 3.6), sharey=True)
with torch.no_grad():
    lines = {
        "seasonal naive": xva[picks, -DAY:],
        "linear": results["linear"]["models"][0](xva[picks]),
        "GRU(64)": results["GRU(64)"]["models"][0](xva[picks]),
    }
for ax, i in zip(axes, range(len(picks))):
    ax.plot(range(-DAY, 0), xva[picks][i, -DAY:], color="0.6", lw=1, label="last day")
    ax.plot(range(H), yva[picks][i], color="k", lw=1.8, label="truth")
    for name, line in lines.items():
        ax.plot(range(H), line[i], lw=1.2, ls="--", label=name)
    ax.axvline(0, color="0.85")
axes[0].set_ylabel("scaled demand")
axes[-1].legend(fontsize=8)
fig.suptitle("three validation days: the last observed day, the truth, three forecasts")
plt.tight_layout()

All three forecasts follow the daily shape; they differ in how far they move the level and
the peak away from yesterday's. The plot is one line per model, and that line is a claim of
certainty that none of these models can back: it says nothing about how far the truth may
fall from it. That is what `01_probabilistic` adds.

## 0.8 The test split, once

Every number above is validation, and the window, the epoch count and the hidden size were
chosen with those numbers in view. The test year is there to find out how much of the
table describes the validation year rather than the problem. It is scored once, here, and
nothing is changed after looking at it.

In [ ]:
test_results = {
    "naive (repeat last)": mae(yte, xte[:, -1:].expand(-1, H)),
    "seasonal naive (1 day)": mae(yte, xte[:, -DAY:]),
}
with torch.no_grad():
    for name in ("linear", "GRU(64)"):
        test_results[name] = float(
            np.mean([mae(yte, m(xte)) for m in results[name]["models"]])
        )

test_order = sorted(test_results, key=lambda k: test_results[k])
print(f"{'model':24s} {'val MAE':>8} {'test MAE':>9} {'val rank':>9} {'test rank':>10}")
print("-" * 64)
for name in order:
    print(
        f"{name:24s} {results[name]['mae']:8.4f} {test_results[name]:9.4f} "
        f"{order.index(name) + 1:9d} {test_order.index(name) + 1:10d}"
    )

The ranking holds on the test year: linear (0.2321), GRU (0.2668), seasonal naive (0.2741),
naive (0.7315), the same order as on validation. The errors themselves move, and not all in
the same direction. Seasonal naive and linear do better on the test year than on validation,
the GRU does worse, and its lead over seasonal naive shrinks from 13% to 3%. A score
describes a model and the period it was measured on: compare scores within one split, and
never call the difference between splits progress.

## What `01_probabilistic` takes from this

- **The series, the window and the split.** `load_ausalia()`, `W = 4 * DAY`, `H = DAY`,
  stride 24 for training and 48 for validation, scaled with the training statistics.
  `01_probabilistic` cuts exactly these windows.
- **The baseline has a name.** On this series it is seasonal naive (validation MAE 0.2906),
  and plain naive (0.7362) is a weak reference. `01_probabilistic` starts from the naive
  forecast plus its residuals; keep in mind which point forecast is the one to beat.
- **The two trunks.** A linear model and a GRU(64), both ahead of seasonal naive, the linear
  model ahead of the GRU at a hundredth of the training time. `01_probabilistic` puts a
  distributional head on both and checks whether that ordering survives once the forecast
  has a width.
- **Seeds and cost.** Three seeds give the `+/-` column; seconds and parameters sit next to
  every score.

## Exercises

**E1 — window length.** `W = 4 * DAY` was chosen to hold four days; nothing above derives
it. Re-run the linear model at `W` of 1, 2, 4 and 7 days. Where does more history stop
helping, and does the answer change for the GRU?

**E2 — a weekly baseline.** With `W = 7 * DAY` the window holds last week. Add a weekly
seasonal naive (`x[:, -7 * DAY:][:, :H]`) and decide again which naive model is the
baseline. Does the MASE of the linear model change when its reference does?

**E3 — epochs.** `EPOCHS = 8` is the value `01_probabilistic` uses. Run the GRU at 4, 8 and
16 epochs with three seeds each. Does its gap to the linear model close with more training,
and what does the cost column say about it?